### MCP File System Implementation

#### Import and Load the packages

In [3]:
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown

from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    print("API Key is missing")

MODEL = "gpt-4o-mini"

In [4]:
# Absolute path is important: the server needs a full path for its allowed directory.
SANDBOX_DIR = os.path.abspath("secret_project_007")
os.makedirs(SANDBOX_DIR, exist_ok=True)

# Drop in a couple of sample files for the agent to find.
# encoding="utf-8" keeps the em dashes (—) safe on Windows (which defaults to cp1252).
with open(os.path.join(SANDBOX_DIR, "notes.txt"), "w", encoding="utf-8") as f:
    f.write(
        "Project Falcon — kickoff notes\n"
        "- Estimated budget: $42,000\n"
        "- Launch target: end of Q3\n"
        "- Owner: Priya\n"
        "- Core team: Marco (engineering), Lena (design), Sam (go-to-market)\n"
        "What it is:\n"
        "Falcon is a tech news and discussion community with AI avatars who provide commentary\n"
        "It's like a Hacker News + a Late Night Talk Show, but in written format and with AI commentators.\n"
        "The bet: engaging AI avatars will quickly become fan-favourites because of the unique dynamic between them.\n"
        "Premise:\n"
        "Ship before the competition announces theirs; quiet until then\n"
        "Hacker News is the benchmark — we study it constantly to understand what the community wants\n"
        "Two competitors rumored to be building something similar — speed is the moat\n"
        "'Falcon' is a placeholder; marketing hates it, but it has stuck\n"
        "Priya's one hard rule: do not slip the Q3 date to add features\n"
        "Open question: what are the most popular categories on Hacker News?\n"
    )

with open(os.path.join(SANDBOX_DIR, "todo.md"), "w", encoding="utf-8") as f:
    f.write(
        "# To do\n"
        "- [X] Book venue\n"
        "- [ ] Confirm budget with finance\n"
        "- [X] Lock the core team\n"
        "- [X] Brief the team on keeping this quiet\n"
        "- [ ] Draft announcement\n"
        "- [ ] Decide whether to keep the 'Falcon' name or rebrand before launch\n"
        "- [ ] Pressure-test the Q3 timeline with engineering\n"
        "- [ ] Scan the Hacker News front page, group top 20 stories into broad categories\n"
    )

print(f"✅ Sandbox ready at: {SANDBOX_DIR}")
print("   Files:", os.listdir(SANDBOX_DIR))

#==================================================

# Create a decoy folder
OTHER_DIR = os.path.abspath("secret_project_006")
os.makedirs(OTHER_DIR, exist_ok=True)

# Feels empty.. Drop in a decoy file.
with open(os.path.join(OTHER_DIR, "decoy_file.txt"), "w") as f:
    f.write("This is a decoy file")

print(f"✅ Decoy folder is ready at: {OTHER_DIR}")
print("   Files:", os.listdir(OTHER_DIR))

✅ Sandbox ready at: /Users/gauravkamath/Documents/Code/ai-engineering/AI-Engineering-Part3/secret_project_007
   Files: ['notes.txt', 'todo.md']
✅ Decoy folder is ready at: /Users/gauravkamath/Documents/Code/ai-engineering/AI-Engineering-Part3/secret_project_006
   Files: ['decoy_file.txt']


In [5]:
!node --version

v26.5.0


In [6]:
!npx --version

11.17.0


### Filesystem MCP Server

In [7]:
filesystem_server_params = {
    "command":"npx",
    "args": ["-y", "@modelcontextprotocol/server-filesystem", SANDBOX_DIR]
}

In [8]:
async with MCPServerStdio(name="File System Server", params=filesystem_server_params, 
                          client_session_timeout_seconds=60) as server2:
    fs_tools = await server2.list_tools()

    print(f"✅ MCP server Connected. The server offers {len(fs_tools)} tool(s):\n")
    for tool in fs_tools:
        print(f"🛠️ {tool.name} --   {tool.description.strip().splitlines()[0]}")

✅ MCP server Connected. The server offers 14 tool(s):

🛠️ read_file --   Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.
🛠️ read_text_file --   Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Operates on the file as text regardless of extension. Only works within allowed directories.
🛠️ read_media_file --   Read a file and return it as a base64-encoded content block with its MIME type. Image and audio files are returned as image/audio content; any other file type is returned as an embedded resource. Only works within allowed directories.
🛠️ read_multiple_files --   Read the contents of multiple files simultaneously. This is more efficien

#### Filesystem Agent(with MCP Server)

In [22]:
FILES_AGENT_PROMPT = f"""You are a file assistant. You work inside the directory {SANDBOX_DIR}.
Always use FULL paths under {SANDBOX_DIR} (e.g. {SANDBOX_DIR}/notes.txt).
You have filesystem tools (via an MCP server) to list, read, search, write, and edit files.
When asked about files, first list or read what's there, then act based on what you actually
find. Be concise, and tell the user exactly which files you read or changed.
When you are searching for any text inside the files, make sure the text is case insensitive
"""

In [10]:
# Read files
async with MCPServerStdio(name="Read Filesystem server", params=filesystem_server_params, 
                          client_session_timeout_seconds=60) as server2:
    files_agent = Agent(
        name = "Files Agent",
        instructions = FILES_AGENT_PROMPT,
        model = MODEL,
        mcp_servers = [server2]
    )

    result = await Runner.run(
        files_agent,
        input = "What files are in my folder and what is each one about? give me one-line summary per file",
        max_turns=10
    )

    print(f"Last Agent: {result.last_agent.name}")
    print("------")
    print(result.final_output)

Last Agent: Files Agent
------
Here are the files in your folder, along with a brief summary of each:

1. **notes.txt**: Contains kickoff notes for Project Falcon, detailing budget, launch timeline, team members, and project concept involving AI avatars in tech discussion.

2. **todo.md**: A to-do list for Project Falcon, outlining completed tasks and remaining actions to prepare for the project's launch, including budget confirmation and branding decisions.


In [ ]:
# Search files
async with MCPServerStdio(name="Read Filesystem server", params=filesystem_server_params, 
                          client_session_timeout_seconds=60) as server2:
    files_agent = Agent(
        name = "Files Agent",
        instructions = FILES_AGENT_PROMPT,
        model = MODEL,
        mcp_servers = [server2]
    )

    result = await Runner.run(
        files_agent,
        input = "Search for every files that mentions notes",
        max_turns=10
    )

    print(f"Last Agent: {result.last_agent.name}")
    print("------")
    print(result.final_output)

Last Agent: Files Agent
------
I found a file that mentions "notes":

- `/Users/gauravkamath/Documents/Code/ai-engineering/AI-Engineering-Part3/secret_project_007/notes.txt`

Would you like to read its contents?


In [29]:
# MCP boundary restrictions.
async with MCPServerStdio(name="Read Filesystem server", params=filesystem_server_params, 
                          client_session_timeout_seconds=60) as server2:
    files_agent = Agent(
        name = "Files Agent",
        instructions = FILES_AGENT_PROMPT,
        model = MODEL,
        mcp_servers = [server2]
    )

    result = await Runner.run(
        files_agent,
        input = f"List all the files inside {OTHER_DIR}",
        max_turns=10
    )

    print(f"Last Agent: {result.last_agent.name}")
    print("------")
    print(result.final_output)

Last Agent: Files Agent
------
I can't access the directory `/Users/gauravkamath/Documents/Code/ai-engineering/AI-Engineering-Part3/secret_project_006`. However, I can assist you with files in the allowed directory `/Users/gauravkamath/Documents/Code/ai-engineering/AI-Engineering-Part3/secret_project_007`. Would you like to see those?


In [32]:
# Create a summary.md file.
# with a summary of the notes.txt and todo.md files.
async with MCPServerStdio(name="Read Filesystem server", params=filesystem_server_params, 
                          client_session_timeout_seconds=60) as server2:
    files_agent = Agent(
        name = "Files Agent",
        instructions = FILES_AGENT_PROMPT,
        model = MODEL,
        mcp_servers = [server2]
    )

    result = await Runner.run(
        files_agent,
        input = "generate a summary.md file that summarizes the notes.txt and todo.md files\
            summarize the outstanding todo list items.",
        max_turns=10
    )

    print(f"Last Agent: {result.last_agent.name}")
    print("------")
    print(result.final_output)

Last Agent: Files Agent
------
I have created a `summary.md` file that includes a concise summary of the kickoff notes and the outstanding to-do list items. Here’s a brief overview of its content:

### Summary of Project Falcon
- **Estimated Budget:** $42,000
- **Launch Target:** End of Q3
- **Owner:** Priya
- **Core Team:** Marco (Engineering), Lena (Design), Sam (Go-to-Market)

### Outstanding To-Do Items
1. Confirm budget with finance
2. Draft announcement
3. Decide whether to keep the 'Falcon' name or rebrand before launch
4. Pressure-test the Q3 timeline with engineering
5. Scan the Hacker News front page and group top 20 stories into broad categories

You can find the file here: [summary.md](sandbox:/Users/gauravkamath/Documents/Code/ai-engineering/AI-Engineering-Part3/secret_project_007/summary.md).
